In [46]:
import pyspark
from pyspark.sql import SparkSession

In [54]:
spark = SparkSession.builder.master("local[*]").appName('test').getOrCreate()

In [56]:
!curl -LO https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  123M  100  123M    0     0  2181k      0  0:00:58  0:00:58 --:--:-- 1610k


In [58]:
!gzip -d fhvhv_tripdata_2021-01.csv.gz

In [59]:
!wc -l fhvhv_tripdata_2021-01.csv

 11908469 fhvhv_tripdata_2021-01.csv


In [60]:
df = spark.read.option("header", "true").csv('fhvhv_tripdata_2021-01.csv')
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [61]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [62]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


In [63]:
!wc -l 10 head.csv

wc: 10: open: No such file or directory
    1001 head.csv
    1001 total


In [64]:
import pandas as pd

In [65]:
df_pandas = pd.read_csv("head.csv")
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [66]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [67]:
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [68]:
from pyspark.sql import types

In [69]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('SR_Flag', types.StringType(), True)
])

In [90]:
df = spark.read.option("header", "true").schema(schema).csv('fhvhv_tripdata_2021-01.csv')

In [91]:
df = df.repartition(24)

In [92]:
df.write.parquet('fhvhv/2021/01', 'overwrite')

26/11/09 23:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/11/09 23:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/11/09 23:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/11/09 23:50:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/11/09 23:50:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/11/09 23:50:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/11/09 23:50:54 WARN MemoryManager: Total allocation exceeds 95.00%

In [74]:
df = spark.read.parquet('fhvhv/2021/01/')

In [93]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [76]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV003') \
  .show()

+---------------+----------------+------------+------------+
|pickup_datetime|dropoff_datetime|PULocationID|DOLocationID|
+---------------+----------------+------------+------------+
+---------------+----------------+------------+------------+



In [77]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


In [78]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-01 16:08:13|2021-01-01 16:11:57|          37|          36|   NULL|
|           HV0003|              B02871|2021-01-02 01:14:25|2021-01-02 01:17:52|         107|          79|   NULL|
|           HV0003|              B02872|2021-01-01 03:24:26|2021-01-01 03:49:26|          17|         129|   NULL|
|           HV0005|              B02510|2021-01-02 13:14:14|2021-01-02 13:37:31|          42|         159|   NULL|
|           HV0003|              B02888|2021-01-03 10:38:10|2021-01-03 11:11:33|          26|         155|   NULL|
|           HV0003|              B02875|2021-01-03 12:58:12|2021-01-03 13:01:33|

In [79]:
from pyspark.sql import functions as F

In [97]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [99]:
df_date = df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
  .withColumn('dropoff_date', F.to_date(df.dropoff_datetime))

In [82]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [83]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [100]:
df_date \
    .withColumn('base_id', crazy_stuff_udf(df_date.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date',
            'PULocationID', 'DOLocationID') \
    .show()

[Stage 29:>                                                         (0 + 1) / 1]

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  s/b3d| 2021-01-01|  2021-01-01|          17|         256|
|  a/b40| 2021-01-04|  2021-01-04|         117|         130|
|  e/a39| 2021-01-04|  2021-01-04|         243|          41|
|  e/9ce| 2021-01-03|  2021-01-03|         106|         195|
|  e/b35| 2021-01-02|  2021-01-02|          82|         198|
|  a/b40| 2021-01-03|  2021-01-03|         141|         129|
|  e/b35| 2021-01-03|  2021-01-03|          23|         156|
|  e/9ce| 2021-01-03|  2021-01-03|          42|         198|
|  e/acc| 2021-01-03|  2021-01-03|         263|          68|
|  s/b3d| 2021-01-01|  2021-01-01|          82|         255|
|  e/b3f| 2021-01-02|  2021-01-02|         169|          47|
|  a/b40| 2021-01-01|  2021-01-01|          41|         168|
|  e/b47| 2021-01-04|  2021-01-04|         116|         142|
|  e/b30| 2021-01-01|  2